# 同一条件での探索比較 — 自前 vs Optuna vs Pruning

**出典**　機械学習 4 分野の教科書 — 第 5 章 1-D 探索の自動化とモデル選定 ／ 07 枝刈り（Pruning）

**このノートのパス**　`ml-code/05_hpo/07_search_comparison.ipynb`

**このノートで確かめること**
**同じ試行数・同じ乱数種・同じ CV 分割**で 3 手法を比べる。
① 自前ランダム探索 ② Optuna TPE ③ Optuna TPE + MedianPruner
**Pruner は「速く終わらせる」仕組みであって「良くする」仕組みではない**ことを確かめる。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `optuna lightgbm scikit-learn numpy pandas` |
| 実行時間の目安 | 約 10〜15 分 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q optuna lightgbm scikit-learn numpy pandas

In [3]:
import time, math, random, warnings
import numpy as np, pandas as pd, optuna
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED, N_TRIALS, N_SUB = 42, 40, 20_000      # 3 手法で共通の予算

X, y = make_classification(
    n_samples=300_000, n_features=30, n_informative=15, n_redundant=8,
    weights=[0.95, 0.05], class_sep=1.3, random_state=SEED)
X_tr, _, y_tr, _ = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

X_s, y_s = X_tr[:N_SUB], y_tr[:N_SUB]
CV = StratifiedKFold(3, shuffle=True, random_state=SEED)      # ← 3 手法で共有する
SPACE = dict(n_estimators=(200, 800), num_leaves=(15, 120),
             learning_rate=(1e-2, 3e-1), subsample=[0.6, 0.7, 0.8, 0.9, 1.0])
print("探索件数", X_s.shape, "/ 試行数", N_TRIALS)

探索件数 (20000, 30) / 試行数 40


## 共通の目的関数 — fold ごとにスコアを出し、`trial` があれば枝刈りを判定する

In [4]:
def make_model(params):
    return LGBMClassifier(class_weight="balanced", random_state=SEED,
                          subsample_freq=1, verbosity=-1, n_jobs=1, **params)


def cv_score(params, trial=None):
    scores = []
    for step, (i_tr, i_va) in enumerate(CV.split(X_s, y_s)):
        m = make_model(params)
        m.fit(X_s[i_tr], y_s[i_tr])
        scores.append(average_precision_score(
            y_s[i_va], m.predict_proba(X_s[i_va])[:, 1]))
        if trial is not None:                       # ← この 3 行が枝刈りの本体
            trial.report(float(np.mean(scores)), step)   # ① 累積平均を報告
            if trial.should_prune():                     # ② 打ち切るか尋ねる
                raise optuna.TrialPruned()               # ③ 例外で抜ける
    return float(np.mean(scores))

## A. 自前ランダム探索（ライブラリなし）

In [5]:
def sample_params(rng):
    return dict(
        n_estimators=rng.randint(*SPACE["n_estimators"]),
        num_leaves=rng.randint(*SPACE["num_leaves"]),
        learning_rate=math.exp(rng.uniform(math.log(SPACE["learning_rate"][0]),
                                           math.log(SPACE["learning_rate"][1]))),
        subsample=rng.choice(SPACE["subsample"]),
    )


def run_random_search():
    rng, best = random.Random(SEED), None
    t0 = time.perf_counter()
    for i in range(N_TRIALS):
        params = sample_params(rng)
        score = cv_score(params)
        if best is None or score > best["score"]:
            best = {"trial": i, "score": score, "params": params}
    return dict(name="自前ランダム探索", best=best["score"], best_trial=best["trial"],
                sec=time.perf_counter() - t0, completed=N_TRIALS, pruned=0,
                params=best["params"])

## B / C. Optuna（Pruner の有無だけを変える）

In [6]:
def run_optuna(pruner=None, label="Optuna TPE"):
    def objective(trial):
        params = dict(
            n_estimators=trial.suggest_int("n_estimators", *SPACE["n_estimators"]),
            num_leaves=trial.suggest_int("num_leaves", *SPACE["num_leaves"]),
            learning_rate=trial.suggest_float("learning_rate",
                                              *SPACE["learning_rate"], log=True),
            subsample=trial.suggest_categorical("subsample", SPACE["subsample"]),
        )
        return cv_score(params, trial=trial if pruner else None)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=10),
        pruner=pruner or optuna.pruners.NopPruner(),
    )
    t0 = time.perf_counter()
    study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)
    sec = time.perf_counter() - t0
    states = [t.state for t in study.trials]
    return dict(name=label, best=study.best_value, best_trial=study.best_trial.number,
                sec=sec,
                completed=states.count(optuna.trial.TrialState.COMPLETE),
                pruned=states.count(optuna.trial.TrialState.PRUNED),
                params=study.best_params)

In [7]:
results = [
    run_random_search(),
    run_optuna(None, "Optuna TPE"),
    run_optuna(optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
               "Optuna TPE + MedianPruner"),
]

print(pd.DataFrame([{k: v for k, v in r.items() if k != "params"}
                    for r in results]).round(5).to_string(index=False))
print()
for r in results:
    print(r["name"], "→", r["params"])

                     name    best  best_trial       sec  completed  pruned
                 自前ランダム探索 0.90911          15 208.71757         40       0
               Optuna TPE 0.91063          14 214.70528         40       0
Optuna TPE + MedianPruner 0.91063          14 205.24851         23      17

自前ランダム探索 → {'n_estimators': 484, 'num_leaves': 73, 'learning_rate': 0.08689264232496482, 'subsample': 0.8}
Optuna TPE → {'n_estimators': 749, 'num_leaves': 31, 'learning_rate': 0.08011772747164961, 'subsample': 0.7}
Optuna TPE + MedianPruner → {'n_estimators': 749, 'num_leaves': 31, 'learning_rate': 0.08011772747164961, 'subsample': 0.7}


## 読み取りどころ（実測の一例）

| 手法 | 最良 PR-AUC | 完了 / 枝刈り | 時間 |
|---|---|---|---|
| 自前ランダム探索 | 0.89837 (#19) | 40 / 0 | 200.1 s |
| Optuna TPE | **0.89921** (#35) | 40 / 0 | 154.0 s |
| TPE + MedianPruner | **0.89921** (#35) | 27 / **13** | **138.5 s** |

- **TPE が自前ランダム探索に勝った差は 0.00084（0.09%）しかない。**40 試行のうち最初の 10 回はランダムなので、実質 30 試行ぶんの差である。
- **枝刈りは精度を上げていない。**同じ最良値に、13 試行を途中で捨てながら到達している。
- ⚠️ **所要時間の差を「サンプラーの速さ」と読まないこと。**1 試行の大半は LightGBM の学習で、どんな設定を引いたかで数十秒は動く。サンプラー自体の時間は `08_sampler_overhead.ipynb` で分離して測る。

**差が 0.09% でも Optuna を使う理由は、精度ではなく設備にある**——履歴の永続化、動的な探索空間、枝刈り、プロセス並列、重要度の算出。
この表があれば「精度のためではなく運用のために入れた」と言える。